In [1]:
import numpy as np
import pandas as pd

DATA_PATH = "/kaggle/input/datasets/shauryapandey19/pm25-dataset/delhi_33station_pm25_spatiotemporal_dataset_2018_2026.csv"

use_columns = [
    "timestamp_ist", "station_name", "latitude", "longitude",
    "temperature_C", "relative_humidity_pct", "wind_speed_m_s", "boundary_layer_height_m",
    "wind_dir_sin", "wind_dir_cos", "pm25", "pm25_original",
]
df = pd.read_csv(DATA_PATH, usecols=use_columns, parse_dates=["timestamp_ist"])

station_list = sorted(df["station_name"].unique())
n_stations = len(station_list)
station_to_idx = {}
for i in range(n_stations):
    station_to_idx[station_list[i]] = i
s_idx = df["station_name"].map(station_to_idx).to_numpy()

full_index = pd.date_range(df["timestamp_ist"].min(), df["timestamp_ist"].max(), freq="h")   # complete hourly index, so a shift of n rows = n hours
n_hours = len(full_index)
t_idx = full_index.get_indexer(df["timestamp_ist"])

assert not df.duplicated(["station_name", "timestamp_ist"]).any()   # grids need one row per (station, hour)
assert (t_idx >= 0).all()

def to_grid(values):
    grid = np.full((n_hours, n_stations), np.nan)   # grid[t, i] = value of station i at hour t
    grid[t_idx, s_idx] = values
    return grid

pm25_grid = to_grid(df["pm25"].to_numpy())           # interpolated pm25, neighbour input only
orig_grid = to_grid(df["pm25_original"].to_numpy())  # real measurements, target only
blh_grid = to_grid(df["boundary_layer_height_m"].to_numpy())
ws_grid = to_grid(df["wind_speed_m_s"].to_numpy())
temp_grid = to_grid(df["temperature_C"].to_numpy())
rh_grid = to_grid(df["relative_humidity_pct"].to_numpy())
wsin_grid = to_grid(df["wind_dir_sin"].to_numpy())
wcos_grid = to_grid(df["wind_dir_cos"].to_numpy())

required_columns = ["pm25_original", "boundary_layer_height_m", "wind_speed_m_s", "wind_dir_sin", "wind_dir_cos"]
row_ok = df[required_columns].notna().all(axis=1).to_numpy()
valid_grid = np.zeros((n_hours, n_stations), dtype=bool)
valid_grid[t_idx, s_idx] = row_ok

In [2]:
station_coords = df.drop_duplicates("station_name").set_index("station_name").loc[station_list, ["latitude", "longitude"]]
lat_r = np.radians(station_coords["latitude"].to_numpy())
lon_r = np.radians(station_coords["longitude"].to_numpy())

# haversine km, dist_matrix[j, i] = target j to source i
a_term = (np.sin((lat_r[None, :] - lat_r[:, None]) / 2) ** 2
          + np.cos(lat_r[:, None]) * np.cos(lat_r[None, :]) * np.sin((lon_r[None, :] - lon_r[:, None]) / 2) ** 2)
dist_matrix = 2 * 6371.0 * np.arcsin(np.sqrt(a_term))

POWERS = [1.5, 2.0]
MIN_DIST_KM = 0.5   # floor so near-identical coordinates cannot get huge weight
base_weights = {}
for p in POWERS:
    w = 1.0 / np.maximum(dist_matrix, MIN_DIST_KM) ** p
    np.fill_diagonal(w, 0.0)   # a station never uses itself, this is the leave-one-out part
    base_weights[p] = w

# bearing from target j to source i, radians clockwise from north
d_lon = lon_r[None, :] - lon_r[:, None]
bearing = np.arctan2(
    np.sin(d_lon) * np.cos(lat_r[None, :]),
    np.cos(lat_r[:, None]) * np.sin(lat_r[None, :]) - np.sin(lat_r[:, None]) * np.cos(lat_r[None, :]) * np.cos(d_lon),
)
bearing_sin = np.sin(bearing)
bearing_cos = np.cos(bearing)

LAGS = [1, 3, 6]

def shift_grid(grid, lag):
    out = np.full_like(grid, np.nan)
    out[lag:] = grid[:-lag]   # value at hour t now holds the value from t - lag
    return out

lag_grids = {}
for lag in LAGS:
    lag_grids[lag] = shift_grid(pm25_grid, lag)

# trailing 24h mean of each station, current hour included, needs 12 real hours
roll_grid = pd.DataFrame(pm25_grid).rolling(24, min_periods=12).mean().to_numpy()

# test rows: target valid, and some OTHER station has pm25 at t, t-1, t-3 and t-6 (same rule as the LGBM LOSO)
avail_list = [~np.isnan(pm25_grid)]
for lag in LAGS:
    avail_list.append(~np.isnan(lag_grids[lag]))

eval_mask = np.zeros((n_hours, n_stations), dtype=bool)
for j in range(n_stations):
    keep = valid_grid[:, j].copy()
    for avail in avail_list:
        others = avail.sum(axis=1).astype(int) - avail[:, j].astype(int)
        keep &= others > 0
    eval_mask[:, j] = keep

print(int(eval_mask.sum()))

2229994


In [3]:
def idw_pred(value_grid, weights):
    ok = ~np.isnan(value_grid)
    w = np.where(ok, weights, 0.0)   # sources without a value get zero weight; weights is (S,) or (T, S)
    num = (np.where(ok, value_grid, 0.0) * w).sum(axis=1)
    den = w.sum(axis=1)
    out = np.full(len(value_grid), np.nan)
    good = den > 0
    out[good] = num[good] / den[good]   # weighted mean, sum(w*v) / sum(w)
    return out

def metrics(y_true, y_pred):
    err = y_pred - y_true
    sse = (err ** 2).sum()
    sst = ((y_true - y_true.mean()) ** 2).sum()
    return {
        "mae": np.abs(err).mean(),
        "rmse": np.sqrt((err ** 2).mean()),
        "r2": 1 - sse / sst,
        "bias": err.mean(),
    }

results = []
station_results = {}

def run_variant(label, power, value_grid, extra_weight=None):
    all_true = []
    all_pred = []
    rows = []
    for j in range(n_stations):
        w = base_weights[power][j]
        if extra_weight is not None:
            w = w[None, :] * extra_weight(j)   # per-hour weights, shape (n_hours, n_stations)
        pred = idw_pred(value_grid, w)
        keep = eval_mask[:, j] & np.isfinite(pred)
        if keep.sum() == 0:
            continue
        y_true = orig_grid[keep, j]
        y_pred = pred[keep]
        all_true.append(y_true)
        all_pred.append(y_pred)
        row = metrics(y_true, y_pred)
        row["station"] = station_list[j]
        rows.append(row)
    overall = metrics(np.concatenate(all_true), np.concatenate(all_pred))   # pooled over all stations, like the LOSO tables
    overall["variant"] = label
    overall["power"] = power
    overall["n"] = int(sum(len(a) for a in all_true))
    results.append(overall)
    station_results[(label, power)] = pd.DataFrame(rows)

In [4]:
for p in POWERS:
    run_variant("same-hour IDW", p, pm25_grid)

In [5]:
for p in POWERS:
    for lag in LAGS:
        run_variant("neighbour lag " + str(lag) + "h", p, lag_grids[lag])
    run_variant("neighbour roll24", p, roll_grid)   # n can be slightly smaller, rows without enough hours are skipped

In [6]:
def wind_factor(j, kappa):
    # target's wind comes FROM wind_dir, alignment +1 = source directly upwind
    alignment = wsin_grid[:, j][:, None] * bearing_sin[j][None, :] + wcos_grid[:, j][:, None] * bearing_cos[j][None, :]
    return np.nan_to_num(np.exp(kappa * alignment), nan=1.0)

for kappa in [0.5, 1.0, 2.0]:
    for p in POWERS:
        run_variant("IDW x wind (kappa=" + str(kappa) + ")", p, pm25_grid, lambda j: wind_factor(j, kappa))

In [7]:
weather_grids = {"BLH": blh_grid, "wind speed": ws_grid, "temperature": temp_grid, "humidity": rh_grid}
weather_std = {}
for name in weather_grids:
    weather_std[name] = np.nanstd(weather_grids[name])

def weather_factor(j, names, beta):
    factor = np.ones((n_hours, n_stations))
    for name in names:
        g = weather_grids[name]
        diff = (g - g[:, j][:, None]) / weather_std[name]   # standardised gap between source and target weather
        factor *= np.nan_to_num(np.exp(-beta * diff ** 2), nan=1.0)
    return factor

weather_sets = {
    "BLH": ["BLH"],
    "wind speed": ["wind speed"],
    "temperature": ["temperature"],
    "humidity": ["humidity"],
    "all four": ["BLH", "wind speed", "temperature", "humidity"],
}
BETA = 1.0
for set_name in weather_sets:
    for p in POWERS:
        run_variant("IDW x weather (" + set_name + ")", p, pm25_grid, lambda j: weather_factor(j, weather_sets[set_name], BETA))

In [8]:
summary = pd.DataFrame(results)[["variant", "power", "n", "mae", "rmse", "r2", "bias"]]
summary = summary.sort_values("mae").reset_index(drop=True).round(3)
summary

,variant,power,n,mae,rmse,r2,bias
0,IDW x weather (BLH),1.5,2229994,24.042,42.321,0.814,0.813
1,IDW x weather (humidity),1.5,2229994,24.053,42.338,0.813,0.833
2,same-hour IDW,1.5,2229994,24.055,42.336,0.814,0.813
3,IDW x weather (temperature),1.5,2229994,24.055,42.339,0.813,0.820
4,IDW x weather (all four),1.5,2229994,24.068,42.375,0.813,0.894
5,IDW x weather (wind speed),1.5,2229994,24.076,42.377,0.813,0.875
6,IDW x wind (kappa=0.5),1.5,2229994,24.176,42.498,0.812,0.971
7,IDW x weather (BLH),2.0,2229994,24.410,43.086,0.807,0.882
8,same-hour IDW,2.0,2229994,24.411,43.086,0.807,0.889
9,IDW x weather (temperature),2.0,2229994,24.414,43.092,0.807,0.893


In [9]:
a = station_results[("same-hour IDW", 1.5)][["station", "mae", "rmse", "r2"]]
b = station_results[("same-hour IDW", 2.0)][["station", "mae", "rmse", "r2"]]
per_station = a.merge(b, on="station", suffixes=("_p1.5", "_p2"))
per_station.round(3).sort_values("r2_p1.5")

,station,mae_p1.5,rmse_p1.5,r2_p1.5,mae_p2,rmse_p2,r2_p2
10,ihbas_dilshad_garden,39.532,61.725,0.294,41.960,66.322,0.185
21,nsut,37.718,58.951,0.406,37.504,58.476,0.416
28,shadipur,38.532,59.925,0.468,39.266,61.180,0.446
4,aya_nagar,33.754,51.058,0.522,32.469,49.417,0.553
17,najafgarh,28.548,43.795,0.671,28.216,42.842,0.685
8,dtu,30.899,55.562,0.682,31.787,57.630,0.657
9,igi_airport_t3,26.143,39.787,0.748,25.512,38.748,0.761
3,aurobindo_marg,21.943,36.538,0.784,21.776,37.003,0.779
19,nehru_nagar,25.325,54.431,0.785,25.554,54.817,0.782
25,pusa,25.681,40.562,0.790,31.050,48.917,0.695


In [10]:
overall_idw = summary[summary["variant"] == "same-hour IDW"].sort_values("power")
print(overall_idw.to_string(index=False))

      variant  power       n    mae   rmse    r2  bias
same-hour IDW    1.5 2229994 24.055 42.336 0.814 0.813
same-hour IDW    2.0 2229994 24.411 43.086 0.807 0.889
